# Decision Trees

Decision trees split the training data based on certain conditions given by the features to produce simple true/false conditions that determine the predicted output. They are traditionally used for classification, but they can also be used for regression.

## Classification: Building the Tree

We want to construct the tree so that we don't just know what conditions to create, but also how to order them so that the top node conditions are the most significant ones. The standard way to measure this is with the Gini impurity measure. It is 1 minus the sum of the squared probabilities that an item belongs in a specific category ($C$ total categories) depending on the condition:

$$
Gini = 1 - \sum_{i=1}^C (p_i)^2
$$

This Gini is computed for the results of each condition. For example, let's say we wanted to see if humidity determines rain (where our feature is just a True/False for isHumid). In our data, when it is humid, it rained 3 days and didn't for 2 days. When it isn't humid, it rained for 1 day and didn't for 3. 

Then, our Gini for humidity being true is $1 - (\frac{3}{5})^2 - (\frac{2}{5})^2 = \frac{12}{25} = 0.48$

Our Gini for humidity being false is $1 - (\frac{1}{4})^2 - (\frac{3}{4})^2 = \frac{6}{16} = 0.375$

To compute the overall Gini of the humidity variable, we take the weighted average of the two cases (true/false) based on the amount of examples that fit each. Humidity being true would be weighted $5/9$ and it being false would be weighted $4/9$.

$$
Gini(Humidity) = 0.48 \cdot \frac{5}{9} + 0.375 \cdot \frac{4}{9} = 0.433
$$

In general, the lower the Gini is, the "purer" the condition is and the more we would want to split the tree based on that condition. The general equation for a boolean feature is:

$$
Gini(Feature) = \frac{n_{true}}{n} \cdot Gini(True) + \frac{n_{false}}{n} \cdot Gini(False)
$$

$$
Gini(Feature) = \frac{n_{true}}{n} \cdot (1 - \sum_{i=1}^C (P(i | true))^2) + \frac{n_{false}}{n} \cdot (1 - \sum_{i=1}^C (P(i | false))^2)

$$

For categorical data, we can either one-hot encode it to use simple true/false Gini. Or, you can take the Gini of all $k$ cases of a feature, and you have:

$$
Gini(Feature) = \sum_{i=1}^{k} \frac{n_i}{n} Gini(Case_i)
$$

Where $n$ is the total searched test data (can be less than actual total if we are already into the tree), and $n_i$ is the number of examples in the $n$ that satisfy $Case_i$.

However, this expression of the Gini for a non-boolean categorical feature is less used because most decision trees are implemented as binary trees. One alternative is to test all possible combinations of a feature's categories. For example, if we have a color feature with ${Red, Blue, Green}, then we may test a {Red} / {Blue or Green} split or {Red or Blue} / {Green} split.

If a feature has $L$ possible categories, we can distribute them into 2 bins in $2^L$ ways. However, these bins are indistinguishable, so we divide by 2, and we also subtract 1 for the case that one bin is empty. Thus, we are left with $2^{L-1} - 1$ ways to group them into two branches. Thus, this technique is only really feasible for low values of $L$.

Alternatively, we can one-hot encode. This solves the computational challenge but can potentially introduce a memory/space challenge.

### Numerical Features

We've discussed how the Gini is computed and how the split is made for binary/categorical features, but numerical features are slightly more complex. First, we need to sort the example set by the numerical feature (this is an $O(n\log n)$ operation). We sort because we want to make binary dividing lines between each of the numerical values to separate the training data in two. For example, if we have the following tuples (Y = yes, N = no for some output category):

$$
(3, Y), (2, N), (4, Y), (1, N) 
$$

The sorted result is

$$
(1, N), (2, N), (3, Y), (4, Y)
$$

We test each dividing line using the averages of each adjacent pair of examples. So, we would test 1.5, 2.5, and 3.5 as a dividing line.

Now, we have constructed 3 boolean conditions we can test: (Is $Feature \leq 1.5$), (Is $Feature \leq 2.5$), (Is $Feature \leq 3.5$)

We compute the Gini of each of these cases like we did before for the boolean/categorical features, and our choice for the best divider for a given numerical feature is the dividing line with the lowest Gini impurity score.


### Combining Numerical and Categorical

Now that we have Gini scores for all the categorical features and numerical features along lines, we choose the split that has the lowest Gini impurity as our root condition. Then, for each subtree, we complete the same operations of computing the Gini scores on the remaining test cases. In the end, we construct a tree where we can make a prediction by traversing the different conditions.

We also don't separate when the Gini impurity without separating is less than the Gini impurity of any separation.

### Overfitting

Decision trees are often prone to overfitting, so there are a few techniques to deal with this challenge:
- Pruning (which I'll discuss later)
- Requiring a minimum number of samples on any given leaf for a candidate split.
- Requiring a minimum number of samples on a node for it to be split.
- Requiring a impurity threshold threshold, where the decrease in Gini from the split versus not doing the split must exceed some value $\xi$.

## Regression Trees

Traditional decision trees output a specific category or boolean values, but we can also use them to output a specific numeric value. This is where Regression Trees come in.

Similarly to how we created candidate splits for numerical features, we do the same thing in a regression tree. Then, given some dividing value $L$, we can take the average of the true values of all the examples to the left of $L$, and to the right.

Then, we can calculate a sense of the error on both sides. On the left, we calculate the sum of the square of all the differences between the true value of each example and the average. We perform the same action on the right.

With equations: If we have the example set $N$, then define $N_{l}$ as the set of all examples $n < L, n \in N$, and $N_{r}$ as the set of all examples $n > L, n \in N$.

The averages on the left and right are:
$$
avg_l = \frac{1}{|N_l|} \sum_{n \in N_l} n
$$
$$
avg_r = \frac{1}{|N_r|} \sum_{n \in N_r} n
$$

The errors are then: 
$$
err_l = \frac{1}{|N_l|} \sum_{n \in N_l} (n - avg_l)^2
$$
$$
err_r = \frac{1}{|N_r|} \sum_{n \in N_r} (n - avg_r)^2
$$

This sum of the differences is called Sum of Squared Residuals (SSR). Finally, we obtain a final error value for the conditional/candidate split: $err_l + err_r = err(Candidate)$.

We pick the overall candidate with the lowest error to be the root split, and we keep repeating the same operation on the smaller example set based on the subtree location.

The same overfitting techniques mentioned above apply to regression trees. The only modification is for the Gini decrease threshold. For regression trees, we can set a sum of squared residuals (error) decrease threshold, where it must decrease a certain amount to justify splitting an ode.


## Pruning

The purpose of __pruning__ is to prevent overfitting. Specifically, pruning is all about removing leaves so that we make more generalized predictions.

We use pruning to decide how deep we want our tree to be to maximize effectiveness (a tree with too many leaves is overfit). The first step is to calculate the error of a fully completed tree. For regression, we compute the mean squared error of each true value and its predicted value. Then, we can start pruning from the deepest leaves, and recalculating the error for each simplified tree. Naturally, the error will increase as the tree becomes smaller, as we are generalizing the predictions.

Cost-complexity pruning works by adding a penalty for tree complexity that is added onto the SSR error, so that even if the error of a smaller tree is greater, its final complexity score is less/the tree is more favored. 

The general equation for cost-complexity pruning is given by:

$$
R_{\alpha}(T) = R(T) + \alpha |T|
$$

where $\alpha$ is a parameter, $R$ is the training error (the SSR error for regression trees), and $|T|$ is the number of leaves for tree $T$.


As $\alpha$ increases, the score encourages a simpler/more-pruned graph. It's also helpful to notice that when $\alpha=0$, the leaf cost completely goes away, so the most complex tree that minimizes SSR is chosen.

The specific choice for $\alpha$ needs to be made with cross-validation, where we swap out different training and testing data roles to ensure the best general parameter and tree complexity. 

Let's say that $t$ is the root of a subtree in $T$. Then the cost of all the components starting at $t$ is the sum of the costs of all of the leaves in the subtree in $t$. 
$$R(T_t) = \sum_{leaf \in |T|} R(leaf)$$

Pruning the tree at $t$ is only worth it if the decrease in cost ($R(t) - R(T_t)$) exceeds the increase in leaves ($\alpha(|T_t| - 1)$). In other words, we can compute some variable $\alpha_t$ for each node $t \in T$:

$$
\alpha_t := \frac{R(t) - R(T_t)}{|T_t| - 1}
$$

Then, we only prune at variable $t$ if $$\alpha_t \geq \alpha$$
